In [3]:
%run ../_MAIN/Hyperparameters_and_data.ipynb

In [4]:
#Add Engineered features
#Calculations used to get BUI and FWI are based off of https://publications.gc.ca/collections/collection_2016/rncan-nrcan/Fo133-1-424-eng.pdf

def addBUI(X):
    df = X.copy()
    #BUI
    dmc = df['DMC']
    dc = df['DC']

    buiLower = (0.8*dmc*dc)/(dmc+0.4*dc)
    buiHigher = dmc-(1.0-0.8*dc/(dmc+0.4*dc))*(0.92+(0.0114*dmc)**1.7)

    bui = np.where(dmc <= 0.4*dc, buiLower, buiHigher)

    df['BUI'] = bui
    return df

def addFWI(X):
    df = X.copy()
    #FWI
    isi = df['ISI']
    bui = df['BUI']

    fdLower = 0.626*bui**0.809 + 2.0
    fdHigher = 1000.0/(25.0+108.64*np.exp(-0.023*bui))
    Fd = np.where(bui <= 80, fdLower, fdHigher)

    b = 0.1*isi*Fd
    b = np.where(b<1, 1.0, b) #Safety so that we don't try to get log(0) in the next line
    fwi = np.where(b>0.1,np.exp(2.72*(0.434*np.log(b))**0.647),b)
    
    df['FWI'] = fwi
    return df

In [8]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, RobustScaler, OneHotEncoder, FunctionTransformer, StandardScaler
#Setting up Preprocessing Pipelines

months = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
days = ['mon','tue','wed','thu','fri','sat','sun']

categorical_pipe = make_pipeline(
    OrdinalEncoder(categories=[months,days]),
    StandardScaler()
)

numerical_pipe = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

preprocessing_original = make_pipeline(
    ColumnTransformer(
        [
            ("cat", categorical_pipe, categorical_columns),
            ("num", numerical_pipe, numerical_columns),
        ]
    )
)

preprocessing_all = make_pipeline(
    FunctionTransformer(addBUI),
    FunctionTransformer(addFWI),
    ColumnTransformer(
        [
            ("cat", categorical_pipe, categorical_columns),
            ("num", numerical_pipe, numerical_columns + ["BUI", "FWI"]),
        ]
    )
)

preprocessing_fire_weather_index = make_pipeline(
    FunctionTransformer(addBUI),
    FunctionTransformer(addFWI),
    ColumnTransformer(
        [
            ("num", numerical_pipe, ["FFMC", "DMC", "DC", "ISI", "BUI", "FWI"]),
        ]
    )
)

# Removes Categorical data and removes DMC, DC, ISI and BUI as all are used in getting the FWI
preprocessing_very_pruned = make_pipeline(
    FunctionTransformer(addBUI),
    FunctionTransformer(addFWI),
    ColumnTransformer(
        [
            ("num", numerical_pipe, ['X', 'Y', 'FFMC', 'FWI', 'temp', 'RH', 'wind', 'rain']),
        ]
    )
)

In [4]:
from sklearn.linear_model import LinearRegression, RidgeCV, GammaRegressor
from sklearn.dummy import DummyRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.ensemble import RandomForestRegressor

#Preprocessing and models that will be used

preprocessing_pipelines = {
    "ORIGINAL": preprocessing_original,
    "ALL" : preprocessing_all,
    "FIRE WEATHER INDEX ONLY": preprocessing_fire_weather_index,
    "VERY PRUNED (TEST)": preprocessing_very_pruned
}

pipeline_features = {
    "ORIGINAL": categorical_columns + numerical_columns,
    "ALL": categorical_columns + numerical_columns  + ["BUI", "FWI"],
    "FIRE WEATHER INDEX ONLY": ["FFMC", "DMC", "DC", "ISI", "BUI", "FWI"],
    "VERY PRUNED (TEST)": ['X', 'Y', 'FFMC', 'FWI', 'temp', 'RH', 'wind', 'rain']
}

models = {
    "Mean" : DummyRegressor(strategy="mean"),
    "Median" : DummyRegressor(strategy="median"),
    "Linear Regression": LinearRegression(),
    "Ridge" : RidgeCV(alphas=[0.01,0.1,1,10,100]),
    # Tried changing KNN weights to "Distance", small negative impact on results
    "KNN n = 3" : KNeighborsRegressor(n_neighbors=3), # This one performs especially poorly, as expected.
    "KNN n = 7" : KNeighborsRegressor(n_neighbors=7),
    "KNN n = 21" : KNeighborsRegressor(n_neighbors=21),
    "Random Forest": RandomForestRegressor(max_depth=2, random_state=RANDOM_STATE),
    "RF max_depth = 5": RandomForestRegressor(max_depth=5, random_state=RANDOM_STATE),
    "RF max_depth = 10": RandomForestRegressor(max_depth=10, random_state=RANDOM_STATE) # Long run time, no positive impact, possibly good idea to discard
#    "Gamma Regressor": GammaRegressor() Doesn't work due to the 0s, normalisation fixes?
}